In [2]:
import pandas as pd

try:
    df_link = pd.read_csv('movie_cast.csv')
    df_movies = pd.read_csv('movies.csv')
    df_cast = pd.read_csv('casts.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"Kích thước file liên kết gốc: {df_link.shape}")

trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'cast_id', 'cast_role'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id, cast_role): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'cast_id', 'cast_role'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")

# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có cast_id nào trong file liên kết mà KHÔNG tồn tại bên file cast không
cast_mo_coi = df_link[~df_link['cast_id'].isin(df_cast['cast_id'])]
print(f"Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): {len(cast_mo_coi)}")

# 1. Tìm các bộ phim KHÔNG CÓ diễn viên nào đóng trong bảng liên kết
movies_khong_cast = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print(f"Số lượng phim bị 'trống' diễn viên: {len(movies_khong_cast)}")

# 2. Tìm các diễn viên KHÔNG THAM GIA bộ phim nào trong bảng liên kết
cast_khong_movie = df_cast[~df_cast['cast_id'].isin(df_link['cast_id'])]
print(f"Số lượng diễn viên không đóng phim nào: {len(cast_khong_movie)}")

# BIỆN PHÁP XỬ LÝ (Tùy chọn): Lọc bỏ các dòng "mồ côi" này để tránh lỗi Foreign Key khi nạp SQL
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['cast_id'].isin(df_cast['cast_id'])]
print(f"Kích thước file liên kết sau khi làm sạch hoàn toàn: {df_link.shape}")
# 1. Chỉ giữ lại những phim thực sự xuất hiện trong bảng liên kết cast_movie
df_movies_cleaned = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]
# 2. Chỉ giữ lại những diễn viên thực sự xuất hiện trong bảng liên kết cast_movie
df_cast_cleaned = df_cast[df_cast['cast_id'].isin(df_link['cast_id'])]
# 3. Xuất ra các file sạch cuối cùng để nạp vào SQL
#df_movies_cleaned.to_csv("movies_final.csv", index=False)
#df_cast_cleaned.to_csv("casts_final.csv", index=False)
# Đổi tên cột id của cast nếu cần, ví dụ: df_cast_cleaned.to_csv("cast_final.csv", index=False)

print(f"Đã làm sạch! Số phim còn lại: {len(df_movies_cleaned)}, Số diễn viên còn lại: {len(df_cast_cleaned)}")

# Lưu lại file liên kết sạch
df_link.to_csv("movie_cast_cleaned.csv", index=False)
#print("Đã xuất file sạch 'movie_cast_cleaned.csv' sẵn sàng nạp SQL!")



Đọc thành công các file dữ liệu!
Kích thước file liên kết gốc: (243885, 3)
Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id, cast_role): 0
Kích thước sau khi xóa trùng lặp cặp: (243885, 3)
Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 4948
Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): 0
Số lượng phim bị 'trống' diễn viên: 0
Số lượng diễn viên không đóng phim nào: 0
Kích thước file liên kết sau khi làm sạch hoàn toàn: (238937, 3)
Đã làm sạch! Số phim còn lại: 4520, Số diễn viên còn lại: 112583


In [4]:
import pandas as pd

# 1. Đọc file liên kết và các file thực thể (Hãy đổi tên file .csv cho đúng với máy bạn)
try:
    df_link = pd.read_csv('movie_genre.csv')
    df_movies = pd.read_csv('movies.csv')  # Sử dụng file movies đã làm sạch ở bước trước
    df_genre = pd.read_csv('genre.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"Kích thước file liên kết gốc MOVIE_GENRE: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên cả cặp khóa ngoại (theo đúng sơ đồ ERD của bạn)
# Lưu ý: subset dùng đúng tên cột trong file của bạn (ví dụ: 'movie_id' và 'gende_id' hoặc 'genre_id')
trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'genre_id'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cặp (movie_id, gende_id): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'genre_id'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")
# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có gende_id nào trong file liên kết mà KHÔNG tồn tại bên file genre không
genre_mo_coi = df_link[~df_link['genre_id'].isin(df_genre['genre_id'])]
print(f"Số dòng chứa 'gende_id' lạ (không có trong bảng Genre): {len(genre_mo_coi)}")
# --- BƯỚC 3: LỌC BỎ DỮ LIỆU MỒ CÔI VÀ CÔ LẬP ĐỂ SẠCH DATA 100% ---
# 1. Loại bỏ các dòng mồ côi trong bảng liên kết để tránh lỗi Foreign Key trong SQL
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['genre_id'].isin(df_genre['genre_id'])]
print(f"Kích thước file liên kết sau khi drop mồ côi: {df_link.shape}")
movies_khong_genre = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print("Movie không có genre", movies_khong_genre.shape)
# 2. Loại bỏ các thể loại phim không có bất kỳ bộ phim nào thuộc về nó (Dữ liệu cô lập)
df_genre_cleaned = df_genre[df_genre['genre_id'].isin(df_link['genre_id'])]
print(df_genre_cleaned.shape)
# 3. Loại bỏ các bộ phim không có thể loại nào (Nếu tiêu chí hệ thống khuyến nghị yêu cầu)
df_movies_cleaned = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]
print(df_movies_cleaned.shape)
df_link.to_csv("movie_genre_cleaned.csv", index=False)

Đọc thành công các file dữ liệu!
Kích thước file liên kết gốc MOVIE_GENRE: (12598, 2)
Số lượng dòng bị trùng lặp cặp (movie_id, gende_id): 0
Kích thước sau khi xóa trùng lặp cặp: (12598, 2)
Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 683
Số dòng chứa 'gende_id' lạ (không có trong bảng Genre): 0
Kích thước file liên kết sau khi drop mồ côi: (11915, 2)
Movie không có genre (0, 5)
(23, 2)
(4520, 5)


In [15]:
df_movies = pd.read_csv("movies.csv")
df_link = pd.read_csv("movie_keyword.csv")
df_movies = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]
print(df_movies.shape)
df_movies.to_csv("movies.csv")

(0, 6)


In [22]:
df_keyword = pd.read_csv("keyword.csv")
df_link = pd.read_csv("movie_keyword.csv")
df_keyword = df_keyword[~df_keyword['keyword_id'].isin(df_link['keyword_id'])]
print(df_keyword.shape)
#df_keyword.to_csv("keyword.csv")


(0, 3)


In [26]:
df_cast = pd.read_csv("casts.csv")
df_link = pd.read_csv("movie_cast.csv")
df_cast = df_cast[~df_cast['cast_id'].isin(df_link['cast_id'])]
print(df_cast.shape)
#df_cast.to_csv("casts.csv")

(0, 4)


In [29]:
import os
import pandas as pd
from ydata_profiling import ProfileReport

# 1. Lấy danh sách tất cả các file CSV trong thư mục hiện tại
all_files = [f for f in os.listdir('.') if f.endswith('.csv')]

print(f"Tìm thấy {len(all_files)} file CSV cần xuất báo cáo.")

# 2. Vòng lặp tự động xử lý từng file
for file_name in all_files:
    report_name = file_name.replace('.csv', '_report.html')
    print(f"\n--------------------------------------------------")
    print(f"🔄 Đang xử lý file: {file_name}...")

    try:
        # Đọc dữ liệu
        df = pd.read_csv(file_name)
        print(f"   Kích thước: {df.shape[0]} hàng, {df.shape[1]} cột.")

        # TỐI ƯU CẤU HÌNH:
        # Các file tương tác lớn như ratings.csv hoặc movie_cast.csv rất nặng.
        # Nếu file có trên 50,000 dòng, tự động bật minimal=True để chạy siêu nhanh và không treo máy.
        if df.shape[0] > 50000:
            print("   ⚠️ File có dung lượng lớn, tự động bật Chế độ Tối giản (Minimal Mode) để tránh treo RAM...")
            profile = ProfileReport(df, title=f"Báo cáo - {file_name}", minimal=True)
        else:
            profile = ProfileReport(df, title=f"Báo cáo - {file_name}", explorative=True)

        # Xuất file HTML
        profile.to_file(report_name)
        print(f"   ✅ Đã xuất xong: {report_name}")

    except Exception as e:
        print(f"   ❌ Lỗi khi xử lý file {file_name}: {e}")

print("\n==================================================")
print("🎉 HOÀN THÀNH TOÀN BỘ! Hãy kiểm tra các file HTML mới được tạo trong thư mục.")


Tìm thấy 11 file CSV cần xuất báo cáo.

--------------------------------------------------
🔄 Đang xử lý file: casts.csv...
   Kích thước: 112583 hàng, 3 cột.
   ⚠️ File có dung lượng lớn, tự động bật Chế độ Tối giản (Minimal Mode) để tránh treo RAM...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 254.97it/s]


   ✅ Đã xuất xong: casts_report.html

--------------------------------------------------
🔄 Đang xử lý file: companies_movie.csv...
   Kích thước: 13783 hàng, 2 cột.


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 280.78it/s]


   ✅ Đã xuất xong: companies_movie_report.html

--------------------------------------------------
🔄 Đang xử lý file: company.csv...
   Kích thước: 5374 hàng, 2 cột.


...

   ✅ Đã xuất xong: movie_keyword_report.html

--------------------------------------------------
🔄 Đang xử lý file: ratings.csv...
   Kích thước: 49635 hàng, 4 cột.


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 294.90it/s]


   ✅ Đã xuất xong: ratings_report.html

--------------------------------------------------
🔄 Đang xử lý file: users.csv...
   Kích thước: 943 hàng, 5 cột.


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 195.37it/s]

   ✅ Đã xuất xong: users_report.html

🎉 HOÀN THÀNH TOÀN BỘ! Hãy kiểm tra các file HTML mới được tạo trong thư mục.
